# Íris – Consulta, resposta fundamentada e evidência das fontes

Este caderno demonstra o fluxo completo do protótipo: submissão da consulta, recuperação dos trechos, geração da resposta restrita aos trechos recuperados e evidência das fontes de cada frase.

| Seção | Conteúdo |
|---|---|
| 0 | Ambiente de execução |
| 1 | Consulta de exemplo: submissão, trechos recuperados, entrada do modelo gerador, resposta, verificação das citações e evidência por frase |
| 2 | Conjunto de sete consultas |
| 3 | Resultados da avaliação |

As respostas apresentadas estão registradas em `respostas_medidas.json`. Foram geradas em 05/11/2026 por modelo Claude, com as instruções do protótipo (`gerar.py`). O modelo configurado no protótipo (Qwen, pela plataforma Groq) não foi utilizado; o comando `python demonstrar_resposta.py --ao-vivo` executa a geração com o modelo configurado.

A classificação das frases foi realizada por dois avaliadores automáticos, instâncias independentes de modelo de linguagem. Não houve validação humana. Nenhum modelo foi treinado.

A lógica reside em `demonstrar_resposta.py`; as células deste caderno invocam as funções do script e exibem os resultados.

## Conceitos

| Termo | Definição |
|---|---|
| Consulta | Pergunta submetida ao sistema |
| Trecho (chunk) | Segmento de documento, com documento, página, seção e endereço de origem |
| Recuperação em dois estágios | Busca vetorial dos 50 trechos mais similares, seguida de reordenação por cross-encoder, que seleciona 5 |
| Resposta fundamentada | Resposta restrita ao conteúdo dos trechos recuperados |
| Citação | Indicação, ao final de cada frase, do trecho que a sustenta (`[T1]` a `[T5]`) |
| Recusa | Declaração de insuficiência de informação quando os trechos não respondem à consulta |
| Verificação automática | Conferência, sem uso de modelo, da existência dos trechos citados e da presença de citação em cada frase |
| Avaliador automático | Modelo de linguagem que classifica cada frase como sustentada, parcialmente sustentada ou não sustentada pelo trecho citado, com transcrição da passagem |

In [1]:
# 0. Ambiente
import contextlib, io, json, re           # biblioteca padrão
import consultar                          # recuperação
import gerar                              # instruções, verificação das citações e modelo gerador
import demonstrar_resposta as demo        # demonstração do fluxo completo

banco = demo.ligar_banco()                # conexão somente de leitura com o banco vetorial
print("Banco:", banco)
print("Método de recuperação:", consultar.METODO_PADRAO)

with contextlib.redirect_stderr(io.StringIO()):       # suprime a barra de progresso do carregamento
    consultar.buscar("teste")                         # carrega o modelo de embeddings e o reordenador
print("Modelos de recuperação carregados.")

Banco: PostgreSQL 17.11 com pgvector 0.8.7, tabela iris_trechos, 1368 trechos
Método de recuperação: qwen3+rerank50


Modelos de recuperação carregados.


## 1 Consulta de exemplo

### 1.1 Consulta submetida

In [2]:
perguntas = json.load(open(demo.PERGUNTAS, encoding="utf-8"))["itens"]   # conjunto de avaliação: 40 consultas
q = next(x for x in perguntas if x["id"] == "Q27")                       # consulta de exemplo
pergunta = q["pergunta"]
print(q["id"], "|", demo.TIPOS[q["tipo"]])
print(pergunta)

Q27 | evidência de pesquisa
Tempo extra na prova funciona para aluno com TDAH ou com dificuldade? O que diz a pesquisa?


### 1.2 Trechos recuperados

A função `consultar.buscar` executa a recuperação em dois estágios e retorna os cinco trechos com os metadados de citação.

In [3]:
trechos = consultar.buscar(pergunta)      # método padrão: 50 candidatos, reordenação, 5 trechos
fechados = demo.fechados()                # documentos sem licença aberta
for t in trechos:                         # cada trecho traz os metadados de citação
    print(f"[T{t['posicao']}] {t['doc']} | p. {t['pagina'] or '-'} | {t['secao'][:50]} [{t['idioma']}, {t['tipo_evidencia']}]")
    print("     ", t["url"])
    if t["nao_usar_para"]:                # aviso de uso, transmitido ao modelo gerador com o trecho
        print("      AVISO DE USO:", t["nota_uso"][:150])
    print("      " + demo.inicio_do_trecho(t, fechados))   # início do trecho; texto sem licença aberta não é reproduzido

[T1] NCEO-TOOLKIT-TEMPO-EXTRA-2026 | p. - | Extended Time: Research [en, revisao_narrativa]
      https://publications.ici.umn.edu/nceo/accommodations-toolkit/extended-time-research
      "- Three studies showed the positive effects of extended time for high school students with attention-deficit/hyperactivity disorder (ADHD). In one study, some students’ scores were somewhat higher on a reading test when ..."
[T2] NCEO-TOOLKIT-TEMPO-EXTRA-2026 | p. - | Extended Time: Research [en, revisao_narrativa]
      https://publications.ici.umn.edu/nceo/accommodations-toolkit/extended-time-research
      "- Four studies found that extended time did not impact student performance of students with LD. One study found that students with LD in grades K-5 did not use significantly more time than general education students to f..."
[T3] NCEO-TOOLKIT-TEMPO-EXTRA-2026 | p. - | Extended Time: Research [en, revisao_narrativa]
      https://publications.ici.umn.edu/nceo/accommodations-toolkit/extended-time

### 1.3 Entrada do modelo gerador

O modelo gerador recebe as instruções, os cinco trechos numerados e a consulta. As instruções restringem a resposta ao conteúdo dos trechos e exigem citação por frase.

In [4]:
prompt = gerar.montar_prompt(pergunta, trechos)        # instruções + trechos [T1]..[T5] + consulta
print(gerar.INSTRUCOES)                                 # instruções ao modelo gerador
print()
print("Cabeçalho de cada trecho na entrada do modelo gerador:")
for linha in prompt.splitlines():
    if re.match(r"\[T\d\] ", linha):                    # somente a linha de cabeçalho de cada trecho
        print("  ", linha[:150])
print()
print("Tamanho da entrada:", len(prompt.split()), "palavras")

Você é o Agente Estrategista do Íris, que apoia professores a adaptar atividades para alunos
com TDAH sem reduzir a habilidade avaliada. Responda em português do Brasil.

Regras:
1. Use SOMENTE os trechos em TRECHOS. Não use conhecimento próprio, nem para completar.
2. Toda frase da resposta termina com a citação do trecho que a sustenta, no formato [T1], [T2] etc.
   Uma frase pode citar mais de um trecho: [T1][T3].
3. Se um trecho estiver em inglês, use o conteúdo dele e responda em português, citando-o normalmente.
4. Se um trecho tiver AVISO DE USO, respeite o aviso (ex.: não use cartilha sem referências como prova
   de eficácia; não use norma revogada como regra vigente).
5. Se os trechos não respondem à pergunta, responda exatamente: "Não encontrei nos documentos da base informação suficiente para responder."
   e nada mais. É melhor não responder do que inventar.
6. Diferencie o que é evidência de pesquisa, o que é norma e o que é orientação prática.
7. No máximo 6 frases. Sem 

### 1.4 Resposta gerada

A resposta está registrada em `respostas_medidas.json`. A célula verifica que a lista de trechos utilizada na geração é idêntica à recuperada nesta execução.

In [5]:
gravada = json.load(open(demo.GRAVADAS, encoding="utf-8"))["geradores"][demo.GERADOR][q["id"]]
ids_recuperados = [t["chunk_id"] for t in trechos]      # trechos recuperados nesta execução
print("Lista de trechos idêntica à utilizada na geração:", ids_recuperados == gravada["fontes"])
print("Modelo gerador:", gravada["modelo"])
print()
resposta = gravada["resposta"]
for n, frase in enumerate(demo.frases_da_resposta(resposta), 1):   # uma frase por linha
    print(f"{n}. {frase}")

Lista de trechos idêntica à utilizada na geração: True
Modelo gerador: claude-haiku-5-5

1. Três estudos com alunos do ensino médio com TDAH mostraram efeitos positivos do tempo estendido [T1], enquanto outros três não encontraram impacto significativo no desempenho desses alunos [T2].
2. Para alunos com dificuldades de aprendizagem (LD), os resultados também se dividem: quatro estudos apontaram efeitos positivos [T1] e quatro não encontraram benefício [T2].
3. Na síntese da revisão, a pesquisa sobre tempo estendido é considerada inconclusiva, com benefício para alguns alunos com TDAH ou LD em avaliações de conteúdo [T3].
4. Menos da metade dos alunos com TDAH que tinham acesso ao tempo estendido de fato o utilizou [T1].
5. Como orientação prática, uma cartilha sugere permitir tempo extra para responder perguntas em aula ou em prova [T5].
6. Essa cartilha não tem referências e não serve como evidência de eficácia [T5].


### 1.5 Verificação automática das citações

A verificação é realizada por contagem, sem uso de modelo: existência dos trechos citados, presença de citação em cada frase e ocorrência de recusa.

In [6]:
c = gerar.conferir(resposta, trechos)                   # verificação por contagem, sem uso de modelo
print("Recusa:                               ", c["absteve"])
print("Trechos citados na resposta:          ", [f"T{n}" for n in c["trechos_citados"]])
print("Citações de trecho inexistente:       ", c["citacoes_invalidas"])
print("Frases sem citação:                   ", len(c["frases_sem_citacao"]), "de", c["frases"])
print("Trechos com aviso de uso citados:     ", c["citou_trecho_com_aviso"])

Recusa:                                False
Trechos citados na resposta:           ['T1', 'T2', 'T3', 'T5']
Citações de trecho inexistente:        []
Frases sem citação:                    0 de 6
Trechos com aviso de uso citados:      ['ABDA-GUIA-PROFESSORES-2012']


### 1.6 Evidência das fontes por frase

Para cada frase: trecho citado, classificação atribuída por cada avaliador e passagem do trecho que sustenta a frase. Texto de documento sem licença aberta não é reproduzido.

In [7]:
item = demo.uma_consulta(q, banco)                      # a mesma consulta, montada pelo script
ficha = demo.texto_da_consulta(item)                    # texto completo da consulta
print(ficha[ficha.index("4. EVIDÊNCIA DAS FONTES"):])   # seção de evidência das fontes

4. EVIDÊNCIA DAS FONTES
   frase 1: citação T1 = NCEO-TOOLKIT-TEMPO-EXTRA-2026, p. -; T2 = NCEO-TOOLKIT-TEMPO-EXTRA-2026, p. -
            avaliador A: sustentada | avaliador B: parcialmente sustentada
            passagem no T1: "Three studies showed the positive effects of extended time for high school students with attention-deficit/hyperactivity disorder (ADHD)."
   frase 2: citação T1 = NCEO-TOOLKIT-TEMPO-EXTRA-2026, p. -; T2 = NCEO-TOOLKIT-TEMPO-EXTRA-2026, p. -
            avaliador A: sustentada | avaliador B: sustentada
            passagem no T1: "Four studies involved students with LD and resulted in positive effects."
   frase 3: citação T3 = NCEO-TOOLKIT-TEMPO-EXTRA-2026, p. -
            avaliador A: sustentada | avaliador B: sustentada
            passagem no T3: "The research on the effects of extended time is inconclusive. Some studies found benefit while others found mixed effects, or no effect."
   frase 4: citação T1 = NCEO-TOOLKIT-TEMPO-EXTRA-2026, p. -
           

## 2 Conjunto de sete consultas

São apresentadas sete consultas: a primeira consulta de cada tipo no conjunto de avaliação. O critério de seleção foi definido previamente à execução.

In [8]:
itens = demo.demonstrar()                               # 7 consultas; grava outputs/05-consulta-resposta-fonte.txt
print("\n".join(demo.resumo(itens)))                    # resumo, uma linha por consulta

RESUMO
sustentadas: frases classificadas como sustentadas pelos dois avaliadores
demais: parcialmente sustentadas, classificações divergentes ou classificação descartada

consulta  tipo                                    frases  sustentadas   demais  comentário  citação de trecho  frases sem  recuperação
                                                                                  da fonte        inexistente     citação  reproduzida
Q01       prática docente                              6            2        3           1                  0           0       4 de 5
Q11       Desenho Universal para a Aprendizagem        4            2        1           1                  0           0     idêntica
Q19       norma                                        5            5        0           0                  0           0     idêntica
Q27       evidência de pesquisa                        6            4        1           1                  0           0     idêntica
Q33       neuromito

Na tabela, a coluna "sustentadas" contabiliza as frases classificadas como sustentadas pelos dois avaliadores. A coluna "demais" contabiliza as frases parcialmente sustentadas, com classificações divergentes ou com classificação descartada. As três últimas colunas resultam de verificação automática.

In [9]:
for i in itens:                                         # consulta, trechos, resposta e evidência de cada uma
    print("=" * 100)
    print(demo.texto_da_consulta(i))
    print()

CONSULTA Q01 (prática docente)

1. CONSULTA SUBMETIDA
   Aluno esquece o enunciado no meio da questão de várias etapas: o que fazer sem dar a resposta?

2. TRECHOS RECUPERADOS (método qwen3+rerank50; 5 trechos entregues ao modelo gerador)
   [T1] ZERBATO-MENDES-2018-DUA-INCLUSAO-ESCOLAR | p. 7-8 | Introdução [pt-BR]
        https://revistas.unisinos.br/index.php/educacao/article/download/edu.2018.222.04/60746207/60747396
        "Antes de dar a resposta, permitir que os alunos tenham tempo suficiente para formular uma resposta ou fazer uma pausa para que os alunos registrem suas respostas em um pedaço de papel ou dispositivo eletrônico ou que com..."
   [T2] RBEE-SEBASTIAN-HEREDERO-2020-DIRETRIZES-DUA | p. 4 | 2.2 Os três princípios [pt-BR]
        https://www.scielo.br/j/rbee/a/F5g6rWB3wTZwyBN4LpLgv5C/?format=pdf&lang=pt
        AVISO DE USO: VERSÃO 2.0: não usar para citar número/nome de ponto de verificação (numeração difere da 3.0); usar como fonte de exemplos práticos. Tradução da

Observações:

- Q01: as respostas foram geradas a partir de um índice local de desenvolvimento. Na recuperação reproduzida no PostgreSQL, um dos cinco trechos diverge. No conjunto de 40 consultas, a lista coincide em 39.
- Q11: na frase 4, a passagem transcrita pelos avaliadores não consta no trecho citado, e a classificação foi descartada.
- Q39: o sistema declarou insuficiência de informação para consulta sobre posologia. Os avaliadores classificaram a recusa como indevida, pois um dos trechos menciona ajuste de dose.

## 3 Resultados da avaliação

Resultados para o conjunto de 40 consultas. O protocolo consta em `docs/rag/avaliacao.md`.

| Indicador | Resultado | Referência |
|---|---|---|
| Recall@5 do método adotado | 0,42 (IC 95%: 0,32 a 0,52) | 0,28 sem reordenação; 0,004 em seleção aleatória |
| Consultas com ao menos um trecho relevante entre os cinco | 33 de 38 | 26 de 38 sem reordenação |
| Frases com citação de trecho inexistente | 0 de 211 | Valor esperado: zero |
| Frases de conteúdo sustentadas pelo trecho citado, para os dois avaliadores | 161 de 187 (86%) | Nenhuma classificada como não sustentada pelos dois avaliadores |
| Respostas que atendem à consulta sem frase reprovada | 29 de 38 (76%; IC 95%: 63% a 89%) | 17 de 38 sem ressalva |
| Recusas nas 2 consultas sem resposta na base | 2 de 2 | Nenhuma recusa nas 38 consultas com resposta |

Com o segundo modelo gerador avaliado (Claude Opus): 160 de 175 frases de conteúdo sustentadas (91%) e nenhuma frase com citação de trecho inexistente, em 199.

## Limitações

- O conjunto de referência da recuperação e os avaliadores das respostas são modelos de linguagem; não houve validação humana.
- As respostas não foram geradas pelo modelo configurado no protótipo; o modelo gerador do produto não está definido.
- O desempenho é inferior em consultas de prática docente: a recuperação retorna 10 dos 43 trechos relevantes.